# $\log 2$: historical constructions

Each section constructs a CMF, walks a trajectory with ramanujantools and prints $\delta_N$ at the deepest sample, where $|\log 2-p_N/q_N|=q_N^{-(1+\delta_N)}$ for the reduced $p_N/q_N$. Run the cells in order with `cmfs.py` next to this notebook. A finite-depth $\delta_N$ does not prove a measure bound.

In [1]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

$(s_0,s_1,\ldots)$ is the first column of the walk.

In [1]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx
from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

from cmfs import rhin_log2, huttner_logs, huttner_logs_seeds

sys.set_int_max_str_digits(0)
with ctx.workdps(62_000):  # q_N reaches about 28,000 digits at depth 3000
    LOG2 = arb(2).log()
DEPTHS = list(range(10, 3001, 10))
x0, x1, x2, y0, y1 = sp.symbols("x0 x1 x2 y0 y1")
results = {}


def delta(approximant):
    """delta with |log 2 - p/q| = q^-(1 + delta), for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):  # enough bits to resolve |log 2 - p/q|
        error = abs(LOG2 - arb(p) / q)
        assert error > 0
        return float((-1 - error.log() / arb(q).log()).mid())

## 1. Alladi–Robinson (1980)

J. Reine Angew. Math. **318** (1980); see [Zudilin (2004), §1](https://arxiv.org/html/math/0404523) · $\mu(\log 2)\le4.6221$

${}_2F_1(-1)$ · start $(1,1;2)$ · trajectory $(1,1;2)$ · $p_N/q_N=s_1/[2(s_1-s_0)]$

In [2]:
cmf = pFq(2, 1, -1)
start = {x0: 1, x1: 1, y0: 2}
trajectory = {x0: 1, x1: 1, y0: 2}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [W[1, 0] / (2 * (W[1, 0] - W[0, 0])) for W in walks]
deltas = [delta(r) for r in approximants]
results["Alladi–Robinson (1980)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.282414787


## 2. Rukhadze (1987)

$\mu(\log 2)<3.89139978$

${}_2F_1(-1)$, dual · start $(1,1;2)$ · trajectory $(7,6;14)$ · $p_N/q_N=s_0/(2s_1+2s_0)$

In [3]:
cmf = pFq(2, 1, -1).dual()  # inverse-transpose matrices
start = {x0: 1, x1: 1, y0: 2}
trajectory = {x0: 7, x1: 6, y0: 14}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [W[0, 0] / (2 * W[1, 0] + 2 * W[0, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Rukhadze (1987)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.346072067


## 3. Marcovecchio (2009)

[Acta Arith. **139** (2009)](https://doi.org/10.4064/aa139-2-5) · $\mu(\log 2)<3.57455391$

${}_3F_2(\tfrac12)$ · start $(1,1,1;2,1)$ · trajectory $(7,5,6;2,1)$ · $p_N/q_N=(s_1-s_0)/s_1$

In [4]:
cmf = pFq(3, 2, sp.Rational(1, 2))
start = {x0: 1, x1: 1, x2: 1, y0: 2, y1: 1}
trajectory = {x0: 7, x1: 5, x2: 6, y0: 2, y1: 1}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(W[1, 0] - W[0, 0]) / W[1, 0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Marcovecchio (2009)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.388170433


## 4. Chudnovsky benchmark (Rhin's reconstruction)

[Rhin (1987), pp. 159–160](https://doi.org/10.1007/978-1-4757-4267-1_11) · $\mu(\log 2)\le4.1344$; Rhin calls the attribution to Chudnovsky probable

Moment CMF of $\int_0^1t^j\,(t(1-t))^{e_1}(6t^2-5t+1)^{e_2}(1+t)^{-b}\,dt$ (`rhin_log2`) · start $(0,0,1)$ · trajectory $(9,1,10)$, so depth $k$ is $N=10k$ · $S_j=C_j+R_j\log 2$, $p_N/q_N=-C_0/R_0$

In [5]:
e1, e2, b = sp.symbols("e1 e2 b")
cmf = rhin_log2()
start = {e1: 0, e2: 0, b: 1}
trajectory = {e1: 9, e2: 1, b: 10}
walks = cmf.walk(trajectory, [N // 10 for N in DEPTHS], start)  # depth k is N = 10k
# S_j = C_j + R_j log 2 at the start.
C = Matrix([[0, 1, -sp.Rational(1, 2), sp.Rational(5, 6)]])
R = Matrix([[1, -1, 1, -1]])
approximants = [-(C * W)[0] / (R * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Chudnovsky benchmark (Rhin reconstruction)"] = (DEPTHS, deltas)
print(f"delta at N = 3000: {deltas[-1]:.9f}")

delta at N = 3000: 0.323450419


## 5. Rhin (1987)

[Rhin (1987), Appendix 1](https://doi.org/10.1007/978-1-4757-4267-1_11) · $\mu(\log 2)\le4.0765$

Moment CMF of $\int_0^1t^j\prod_{i=1}^6p_i(t)^{e_i}\,(1+t)^{-b}\,dt$, $0\le j\le13$, with Rhin's $p_1,\ldots,p_6=t(1-t),\ t^2+2t-1,\ 6t^2-5t+1,\ 7t^2-6t+1,\ 13t^2-11t+2,\ 2t^4+3t^3+3t^2-5t+1$ (`rhin_log2_full`, rank 14; its face $e_2=e_4=e_5=e_6=0$ is `rhin_log2`) · start $(0,0,0,0,0,0,1)$ · path $(\lfloor N\sigma_1\rfloor,\ldots,\lfloor N\sigma_6\rfloor,N+1)$ with Rhin's slopes $\sigma=(0.84943,0.02401,0.09100,0.02068,0.01130,0.00179)$ · $S_j=C_j+R_j\log 2$, $p_N/q_N=-C_0/R_0$

The path is a staircase, not a fixed direction, so the walk steps one axis at a time to each of its points. The symbolic matrices in seven variables are too large to walk, so `rhin_log2_full` gives each axis matrix exactly at an integer point.

In [6]:
from flint import fmpq, fmpq_mat

from cmfs import rhin_log2_full

e = sp.symbols("e1:7")
b = sp.Symbol("b")
M = rhin_log2_full()  # M(x, point): the axis matrix M_x at an integer point
slopes = (84943, 2401, 9100, 2068, 1130, 179)  # Rhin's sigma_i, times 10^5
point = {**{x: 0 for x in e}, b: 1}
# S_j = int_0^1 t^j / (1+t) dt = C_j + R_j log 2 at the start, since S_j + S_(j-1) = 1/j.
C, R = [fmpq(0)], [fmpq(1)]
for j in range(1, 14):
    C.append(fmpq(1, j) - C[-1])
    R.append(-R[-1])
state = fmpq_mat([C, R])
approximants = []
for N in range(1, DEPTHS[-1] + 1):
    # Step one axis at a time to the staircase point (floor(N sigma_1), ..., floor(N sigma_6), N + 1).
    target = {**{x: s * N // 10**5 for x, s in zip(e, slopes)}, b: N + 1}
    for x in (*e, b):
        while point[x] < target[x]:
            state = state * M(x, point)
            point[x] += 1
    if N in DEPTHS:
        approximants.append(-state[0, 0] / state[1, 0])
deltas = [delta(r) for r in approximants]
results["Rhin (1987)"] = (DEPTHS, deltas)
print(f"delta at N = 3000: {deltas[-1]:.9f}")

delta at N = 3000: 0.326425438


## 6. Huttner (2003): $\log 2$ from the simultaneous construction

[Corollary 3.1(4), pp. 146–147](https://www.jstage.jst.go.jp/article/kyushujm/57/1/57_1_129/_pdf)
gives a linear-independence exponent $10.1017577\ldots$ for $1,\log2,\log3$.
It implies $\mu(\log 2)\le11.1017577\ldots$, with comparison scale
$\delta_{\rm eq}=1/10.1017577\ldots\approx0.09899267$.

The source integrals are
$$R_{i,n}=\int_{I_i}\frac{[t(t-\tfrac12)(t-\tfrac13)]^{2n}}{(t-1)^{3n}}\,dt,
\quad I_1=[0,\tfrac13],\quad I_2=[\tfrac13,\tfrac12].$$
With $z=1-t$, their common CMF transports the moments of
$$(z-1)^u(z-\tfrac23)^v(z-\tfrac12)^w z^{-m}.$$
This is a **rank-three, four-axis** field on $(u,v,w,m)$, with start
$(2,2,2,3)$ and direction $(2,2,2,3)$. Source index $n$ uses exactly $n-1$ steps.

Let $P_{32},P_{43},Q$ be the seed rows obtained by Laurent integration at
$n=1$. The individual readouts are $(P_{32}W)_0/(QW)_0$ for $\log(3/2)$
and $(P_{43}W)_0/(QW)_0$ for $\log(4/3)$.
Since $\log 2=1\log(3/2)+\log(4/3)$, use
$$P=1P_{32}+P_{43},\qquad \frac{p_n}{q_n}=\frac{(PW_{n-1})_0}{(QW_{n-1})_0}.$$
This is another readout of the same historical simultaneous construction,
not an independent construction or a new record bound.

The reduced-rational $\delta_n$ need not equal $\delta_{\rm eq}$:
the theorem bounds all integer linear forms, whereas reduction can remove
additional factors from a particular scalar approximant's denominator.
All three moment coefficient pairs were checked by exact source integration
at $n=1,\ldots,7$; all six axis pairs were checked at two regular points.
The source-index-1000 error was computed with overlapping Arb intervals at
two precisions. Supporting scripts and reports remain in ignored `local_checks/`.


In [15]:
u, v, w, m = sp.symbols("u v w m")
cmf = huttner_logs()
start = {u: 2, v: 2, w: 2, m: 3}
trajectory = {u: 2, v: 2, w: 2, m: 3}
depths = list(range(10, 1001, 10))  # source indices
P, Q = huttner_logs_seeds(2)
walks = cmf.walk(trajectory, [N - 1 for N in depths], start)
approximants = [(P * W)[0] / (Q * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Huttner (2003), simultaneous"] = (depths, deltas)
print(f"delta at source index 1000: {deltas[-1]:.15f}")


delta at source index 1000: 0.224912323470657


## All constructions

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published bound | Here |
|:--|--:|:--|
| Alladi–Robinson (1980) | $4.6221$ | CMF |
| Chudnovsky (1980s) | $4.1344$ | CMF of Rhin's reconstruction |
| Rhin (1987) | $4.0765$ | CMF |
| Rukhadze (1987) | $3.89139978$ | CMF |
| Marcovecchio (2009) | $3.57455391$ | CMF |
| Huttner (2003), simultaneous | $11.1017577\ldots$ | four-axis CMF; $\delta_{1000}=0.224912323471$; $\delta_{\rm eq}\approx0.09899267$ |

Earlier records, including Rhin's: [Rhin–Viola (1993), introduction](https://www.numdam.org/item/AIF_1993__43_1_85_0/).
